# 🦠 Disease Burden Analysis — Cameroon vs Sub-Saharan Africa
### Malaria, HIV/AIDS & Tuberculosis: Trends, Comparisons & Policy Insights

**Author:** Pantouin Adjinsala  
**Affiliation:** University Lecturer & Civic Tech Contributor, AfricTivistes CitizenLab Cameroon  
**Context:** Sub-Saharan Africa carries a disproportionate share of the global burden of
malaria, HIV/AIDS, and tuberculosis. This notebook uses World Bank open data to analyse
Cameroon's disease burden trajectory, benchmark it against the Sub-Saharan Africa regional
average, and surface insights relevant to health governance and SDG 3 monitoring.

---

## 📌 Contents
1. Setup & Imports
2. Data Collection via World Bank API
3. Data Cleaning & Reshaping
4. Cameroon Disease Burden Overview
5. Cameroon vs SSA Average — Trend Analysis
6. Cross-Disease Comparison & Burden Index
7. Correlation Analysis
8. SDG 3 Target Tracking
9. Export to Excel
10. Discussion & Policy Implications

## 1. Setup & Imports

In [ ]:
!pip install wbgapi openpyxl xlsxwriter -q

import wbgapi as wb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')

CMR_COLOR = '#007749'   # Cameroon green
SSA_COLOR = '#e67e22'   # SSA orange
print('Libraries loaded successfully.')

## 2. Data Collection via World Bank API

| Disease | Indicator | Code |
|---------|-----------|------|
| Malaria | Incidence (per 1,000 at risk) | SH.MLR.INCD.P3 |
| Malaria | Deaths (per 100,000) | SH.MLR.MORT.P3 |
| Malaria | Children sleeping under ITNs (%) | SH.MLR.NETS.ZS |
| HIV | Prevalence, adults 15-49 (%) | SH.DYN.AIDS.ZS |
| HIV | ART coverage (% of PLHIV) | SH.HIV.ARTC.ZS |
| HIV | New HIV infections (per 1,000) | SH.HIV.INCD.ZS |
| TB | Incidence (per 100,000) | SH.TBS.INCD |
| TB | Treatment success rate (%) | SH.TBS.CURE.ZS |
| TB | Mortality excl. HIV (per 100,000) | SH.TBS.MORT |
| System | Health expenditure per capita (USD) | SH.XPD.CHEX.PC.CD |
| System | Physicians (per 1,000) | SH.MED.PHYS.ZS |

In [ ]:
INDICATORS = {
    'SH.MLR.INCD.P3'    : 'Malaria Incidence (per 1,000)',
    'SH.MLR.MORT.P3'    : 'Malaria Deaths (per 100k)',
    'SH.MLR.NETS.ZS'    : 'ITN Usage, Children (%)',
    'SH.DYN.AIDS.ZS'    : 'HIV Prevalence 15-49 (%)',
    'SH.HIV.ARTC.ZS'    : 'ART Coverage (%)',
    'SH.HIV.INCD.ZS'    : 'New HIV Infections (per 1,000)',
    'SH.TBS.INCD'       : 'TB Incidence (per 100k)',
    'SH.TBS.CURE.ZS'    : 'TB Treatment Success (%)',
    'SH.TBS.MORT'       : 'TB Mortality excl. HIV (per 100k)',
    'SH.XPD.CHEX.PC.CD' : 'Health Expenditure per Capita (USD)',
    'SH.MED.PHYS.ZS'    : 'Physicians (per 1,000)',
}

DISEASE = {
    'SH.MLR.INCD.P3'    : 'Malaria',
    'SH.MLR.MORT.P3'    : 'Malaria',
    'SH.MLR.NETS.ZS'    : 'Malaria',
    'SH.DYN.AIDS.ZS'    : 'HIV',
    'SH.HIV.ARTC.ZS'    : 'HIV',
    'SH.HIV.INCD.ZS'    : 'HIV',
    'SH.TBS.INCD'       : 'Tuberculosis',
    'SH.TBS.CURE.ZS'    : 'Tuberculosis',
    'SH.TBS.MORT'       : 'Tuberculosis',
    'SH.XPD.CHEX.PC.CD' : 'Health System',
    'SH.MED.PHYS.ZS'    : 'Health System',
}

ECONOMIES = ['CMR'] + list(wb.region.members('SSF'))

print('Fetching data from World Bank API...')
raw = wb.data.DataFrame(
    list(INDICATORS.keys()),
    economy=ECONOMIES,
    time=range(2000, 2023),
    skipBlanks=True,
    columns='series'
)
print(f'Raw shape: {raw.shape}')
raw.head()

## 3. Data Cleaning & Reshaping

In [ ]:
df = raw.reset_index()
df.columns.name = None
df = df.rename(columns={'economy':'country_code','time':'year'})
df = df.rename(columns=INDICATORS)
df['year'] = df['year'].astype(str).str.extract(r'(\d{4})').astype(int)

# Separate Cameroon and SSA (excluding CMR for clean SSA average)
cmr_df = df[df['country_code'] == 'CMR'].copy()
ssa_df = df[df['country_code'] != 'CMR'].copy()

# Long format for both
ind_cols = list(INDICATORS.values())

def to_long(d, label):
    m = d.melt(id_vars=['country_code','year'], value_vars=ind_cols,
               var_name='indicator', value_name='value')
    m['group'] = label
    m['disease'] = m['indicator'].map({v:DISEASE[k] for k,v in INDICATORS.items()})
    return m.dropna(subset=['value'])

cmr_long = to_long(cmr_df, 'Cameroon')
ssa_avg  = (ssa_df.melt(id_vars=['country_code','year'], value_vars=ind_cols,
                        var_name='indicator', value_name='value')
            .dropna(subset=['value'])
            .groupby(['year','indicator'], as_index=False)['value'].mean())
ssa_avg['country_code'] = 'SSA'
ssa_avg['group']   = 'SSA Average'
ssa_avg['disease'] = ssa_avg['indicator'].map({v:DISEASE[k] for k,v in INDICATORS.items()})

combined = pd.concat([cmr_long, ssa_avg], ignore_index=True)

print(f'Cameroon records : {len(cmr_long):,}')
print(f'SSA avg records  : {len(ssa_avg):,}')
print(f'Combined         : {len(combined):,}')

## 4. Cameroon Disease Burden Overview

Latest available values for all 11 indicators.

In [ ]:
latest_cmr = (cmr_long.sort_values('year')
              .groupby('indicator', as_index=False).last()
              [['indicator','year','value','disease']]
              .rename(columns={'value':'Latest Value','year':'Data Year'})
              .sort_values(['disease','indicator']))
latest_cmr['Latest Value'] = latest_cmr['Latest Value'].round(2)
print('=== CAMEROON — LATEST INDICATOR VALUES ===')
print(latest_cmr.to_string(index=False))

In [ ]:
fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

disease_colors = {'Malaria':'#c0392b','HIV':'#8e44ad','Tuberculosis':'#2980b9','Health System':'#27ae60'}
plot_indicators = [
    ('Malaria Incidence (per 1,000)',  gs[0,0]),
    ('HIV Prevalence 15-49 (%)',       gs[0,1]),
    ('TB Incidence (per 100k)',        gs[0,2]),
    ('ART Coverage (%)',               gs[1,0]),
    ('TB Treatment Success (%)',       gs[1,1]),
    ('Health Expenditure per Capita (USD)', gs[1,2]),
]

for ind, pos in plot_indicators:
    ax  = fig.add_subplot(pos)
    dis = {v:DISEASE[k] for k,v in INDICATORS.items()}.get(ind,'Health System')
    col = disease_colors.get(dis, '#555')

    c_data = combined[(combined['indicator']==ind) & (combined['group']=='Cameroon')].sort_values('year')
    s_data = combined[(combined['indicator']==ind) & (combined['group']=='SSA Average')].sort_values('year')

    ax.plot(c_data['year'], c_data['value'], color=CMR_COLOR, lw=2, marker='o', ms=3, label='Cameroon')
    ax.plot(s_data['year'], s_data['value'], color=SSA_COLOR, lw=2, ls='--', label='SSA Avg')
    ax.set_title(ind, fontsize=8.5, fontweight='bold')
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7)

fig.suptitle('Cameroon vs SSA Average — Key Health Indicators (2000–2022)',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('cameroon_overview.png', bbox_inches='tight')
plt.show()

## 5. Cameroon vs SSA Average — Disease-by-Disease Trends

Deep-dive trend panels for each of the three diseases.

In [ ]:
for disease in ['Malaria','HIV','Tuberculosis']:
    dis_inds = [v for k,v in INDICATORS.items() if DISEASE[k]==disease]
    n = len(dis_inds)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4), sharey=False)
    if n == 1: axes = [axes]

    for ax, ind in zip(axes, dis_inds):
        c_data = combined[(combined['indicator']==ind)&(combined['group']=='Cameroon')].sort_values('year')
        s_data = combined[(combined['indicator']==ind)&(combined['group']=='SSA Average')].sort_values('year')
        ax.fill_between(c_data['year'], c_data['value'], alpha=0.15, color=CMR_COLOR)
        ax.fill_between(s_data['year'], s_data['value'], alpha=0.10, color=SSA_COLOR)
        ax.plot(c_data['year'], c_data['value'], color=CMR_COLOR, lw=2.5, marker='o', ms=4, label='Cameroon')
        ax.plot(s_data['year'], s_data['value'], color=SSA_COLOR, lw=2.5, ls='--', label='SSA Average')
        ax.set_title(ind, fontsize=9, fontweight='bold')
        ax.set_xlabel('Year')
        ax.legend(fontsize=8)

    plt.suptitle(f'{disease} — Cameroon vs SSA Average (2000–2022)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'trend_{disease.lower()}.png', bbox_inches='tight')
    plt.show()

## 6. Cross-Disease Comparison & Composite Burden Index

We build a **Normalised Disease Burden Index (NDBI)** from incidence and mortality
indicators, normalised so that higher = worse burden. This allows cross-disease
comparison on a common scale.

In [ ]:
burden_indicators = [
    'Malaria Incidence (per 1,000)',
    'Malaria Deaths (per 100k)',
    'HIV Prevalence 15-49 (%)',
    'New HIV Infections (per 1,000)',
    'TB Incidence (per 100k)',
    'TB Mortality excl. HIV (per 100k)',
]

# Latest year per indicator for Cameroon and SSA
burden_rows = []
for ind in burden_indicators:
    for grp in ['Cameroon','SSA Average']:
        sub = combined[(combined['indicator']==ind)&(combined['group']==grp)]
        if sub.empty: continue
        last = sub.sort_values('year').iloc[-1]
        burden_rows.append({'indicator':ind,'group':grp,'value':last['value'],'year':last['year']})

burden_df = pd.DataFrame(burden_rows)

# Normalise 0-100 across both groups per indicator
for ind in burden_indicators:
    mask = burden_df['indicator'] == ind
    mn, mx = burden_df.loc[mask,'value'].min(), burden_df.loc[mask,'value'].max()
    if mx > mn:
        burden_df.loc[mask,'norm'] = (burden_df.loc[mask,'value'] - mn)/(mx - mn)*100
    else:
        burden_df.loc[mask,'norm'] = 50

# NDBI = mean of normalised burden indicators
ndbi = burden_df.groupby('group')['norm'].mean().round(1).reset_index()
ndbi.columns = ['Group','NDBI (0=lowest burden, 100=highest)']
print(ndbi.to_string(index=False))

In [ ]:
# Radar / spider chart comparing Cameroon vs SSA
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

labels   = [i.replace(' (per 1,000)','').replace(' (per 100k)','').replace(' 15-49 (%)','') for i in burden_indicators]
N_axes   = len(labels)
angles   = np.linspace(0, 2*np.pi, N_axes, endpoint=False).tolist()
angles  += angles[:1]

fig, ax = plt.subplots(figsize=(8,8), subplot_kw=dict(polar=True))

for grp, color, ls in [('Cameroon', CMR_COLOR, '-'), ('SSA Average', SSA_COLOR, '--')]:
    vals = [burden_df[(burden_df['indicator']==ind)&(burden_df['group']==grp)]['norm'].values
            for ind in burden_indicators]
    vals = [v[0] if len(v)>0 else 0 for v in vals]
    vals += vals[:1]
    ax.plot(angles, vals, color=color, lw=2.5, ls=ls, label=grp)
    ax.fill(angles, vals, color=color, alpha=0.12)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, size=9)
ax.set_yticks([25,50,75,100])
ax.set_yticklabels(['25','50','75','100'], size=7, color='grey')
ax.set_title('Normalised Disease Burden — Cameroon vs SSA Average',
             fontsize=12, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig('radar_burden.png', bbox_inches='tight')
plt.show()

## 7. Correlation Analysis

Does health expenditure or physician density correlate with lower disease burden
in Cameroon's historical data? We explore this using available years.

In [ ]:
# Pivot Cameroon data wide for correlation
cmr_wide = (cmr_long.groupby(['year','indicator'], as_index=False)['value']
             .mean()
             .pivot(index='year', columns='indicator', values='value'))
cmr_wide.columns.name = None
cmr_wide = cmr_wide.reset_index()

corr_cols = [c for c in ind_cols if c in cmr_wide.columns]
corr_matrix = cmr_wide[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, linewidths=0.5,
    ax=ax, cbar_kws={'label':'Pearson r'},
    annot_kws={'size':7}
)
ax.set_title('Correlation Matrix — Cameroon Health Indicators (2000–2022)',
             fontweight='bold')
plt.xticks(rotation=40, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('correlation_matrix.png', bbox_inches='tight')
plt.show()

## 8. SDG 3 Target Tracking

SDG 3 sets specific 2030 targets for malaria, HIV, and TB. We measure
Cameroon's current trajectory against each target.

In [ ]:
SDG_TARGETS = {
    'Malaria Incidence (per 1,000)'    : {'target': None,  'direction': 'down', 'note': 'End malaria epidemic — no single numeric threshold'},
    'HIV Prevalence 15-49 (%)'         : {'target': None,  'direction': 'down', 'note': 'End AIDS epidemic by 2030'},
    'ART Coverage (%)'                 : {'target': 95,    'direction': 'up',   'note': 'UNAIDS 95-95-95 target'},
    'TB Incidence (per 100k)'          : {'target': 10,    'direction': 'down', 'note': 'SDG 3.3 — <10 per 100k by 2030'},
    'TB Treatment Success (%)'         : {'target': 90,    'direction': 'up',   'note': 'WHO End TB strategy target'},
    'Health Expenditure per Capita (USD)': {'target': None, 'direction': 'up',  'note': 'WHO recommends >$86 per capita'},
}

records = []
for ind, meta in SDG_TARGETS.items():
    sub = combined[(combined['indicator']==ind)&(combined['group']=='Cameroon')].sort_values('year')
    if sub.empty: continue
    latest_val  = sub.iloc[-1]['value']
    latest_year = sub.iloc[-1]['year']
    target = meta['target']
    if target is not None:
        if meta['direction']=='up':
            status = 'On track' if latest_val >= target * 0.9 else 'Below target'
        else:
            status = 'On track' if latest_val <= target * 1.1 else 'Above target'
    else:
        status = 'Monitor'
    records.append({
        'Indicator'     : ind,
        'Latest Value'  : round(latest_val, 2),
        'Data Year'     : int(latest_year),
        '2030 Target'   : target if target else 'See note',
        'Status'        : status,
        'Note'          : meta['note']
    })

sdg_df = pd.DataFrame(records)
print('=== SDG 3 TARGET TRACKING — CAMEROON ===')
print(sdg_df[['Indicator','Latest Value','2030 Target','Status']].to_string(index=False))

In [ ]:
# Visual SDG scorecard
fig, ax = plt.subplots(figsize=(11, 4))
ax.axis('off')

status_colors = {'On track':'#d5f5e3','Below target':'#fadbd8','Above target':'#fadbd8','Monitor':'#fef9e7'}
col_widths = [0.38, 0.14, 0.14, 0.14]
headers    = ['Indicator','Latest Value','2030 Target','Status']
table_data = sdg_df[headers].values.tolist()

table = ax.table(
    cellText=table_data,
    colLabels=headers,
    cellLoc='center', loc='center',
    colWidths=col_widths
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.8)

for j, _ in enumerate(headers):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')

for i, row in enumerate(table_data):
    status = row[3]
    color  = status_colors.get(status, 'white')
    for j in range(len(headers)):
        table[i+1, j].set_facecolor(color)

ax.set_title('SDG 3 Target Scorecard — Cameroon', fontweight='bold', fontsize=12, pad=12)
plt.tight_layout()
plt.savefig('sdg_scorecard.png', bbox_inches='tight')
plt.show()

## 9. Export to Excel

In [ ]:
output_file = 'cameroon_disease_burden_report.xlsx'

with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    wb_xl   = writer.book
    hdr_fmt = wb_xl.add_format({'bold':True,'bg_color':'#2c3e50','font_color':'white','border':1,'align':'center'})

    def write_sheet(df, name, col_width=22):
        df.to_excel(writer, sheet_name=name, index=False)
        ws = writer.sheets[name]
        for i, col in enumerate(df.columns):
            ws.write(0, i, col, hdr_fmt)
        ws.set_column(0, len(df.columns)-1, col_width)

    write_sheet(latest_cmr.rename(columns={'value':'Latest Value','year':'Data Year'}), 'Latest Indicators')
    write_sheet(sdg_df,    'SDG 3 Scorecard')
    write_sheet(burden_df[['indicator','group','value','year']].rename(columns={'value':'Value','year':'Data Year'}), 'Burden Index Data')
    write_sheet(cmr_long[['year','indicator','value','disease']].sort_values(['indicator','year']), 'Cameroon Time Series', 20)
    write_sheet(ssa_avg[['year','indicator','value','disease']].sort_values(['indicator','year']), 'SSA Average Time Series', 20)

print(f'Excel report exported: {output_file}')
print('Sheets: Latest Indicators | SDG 3 Scorecard | Burden Index Data | Cameroon Time Series | SSA Average Time Series')

## 10. Discussion & Policy Implications

### Key Findings
- **Malaria** remains Cameroon's heaviest disease burden. Incidence tracks close to the SSA
  average, suggesting structural rather than policy-specific drivers.
- **HIV/AIDS** prevalence in Cameroon has declined since its 2000s peak, but ART coverage
  still falls short of the UNAIDS 95-95-95 target, indicating a treatment gap.
- **Tuberculosis** incidence is declining slowly. Treatment success rates are improving but
  remain below the WHO 90% target — drug-resistant TB is a growing concern.
- **Health expenditure** per capita remains low, and the correlation analysis suggests a
  meaningful relationship between spending and disease outcome improvements.

### Limitations
- World Bank data aggregates national figures — masking significant regional disparities
  (e.g. Far North vs Centre regions in Cameroon).
- Reporting gaps in earlier years can distort trend lines; figures should be interpreted
  as indicative rather than definitive.
- Causal inference is not possible from this observational, time-series data alone.

### Planned Extensions
1. **Sub-national analysis** — integrate DHS and MICS data for region-level disease mapping
2. **Forecasting** — apply ARIMA or Prophet models to project 2030 indicator values
3. **Governance overlay** — correlate disease burden with transparency and public finance
   indicators to explore governance linkages
4. **Interactive dashboard** — deploy as a Streamlit app for ministry or NGO use

---

> *This notebook was developed as part of civic tech and health governance research in Cameroon.*  
> *Data: World Bank Open Data (CC BY 4.0)*

**References**
- World Bank Open Data: https://data.worldbank.org
- WHO Global Health Observatory: https://www.who.int/data/gho
- UNAIDS Data: https://www.unaids.org/en/resources/documents
- SDG 3 targets: https://sdgs.un.org/goals/goal3